In [1]:
import os
import pickle
import pandas as pd

import plotly.graph_objects as go

import mne

from joblib import Parallel, delayed
import re

import sys
# sys.path.insert(0, './')
sys.path.insert(0, '../')
from utils.analysis_helpers import compute_erps, fit_lmm_for_time_bins, plot_erp
from utils.bids_compliance import read_epochs, save_psd_epochs, read_psd_epochs
from utils.analysis_helpers import filter_epochs_by_distance_to_probe, classify_onoff_epochs


print('Packages loaded')

# Paths and settings
# root = "/network/lustre/iss02/cenir/analyse/meeg/CYBERSART/"
# root = "//l2export/iss02.cenimodule r/analyse/meeg/CYBERSART/"
root = "/Volumes/cenir/analyse/meeg/CYBERSART/"


derivatives_folder = os.path.join(root, "derivatives_nico")
subjects = [f"{i:02}" for i in range(2, 43)]
tasks = ['Sart1', 'Sart2', 'Sart3', 'Sart4']
data = "eeg"

# Conditions and settings for classification and evoked generation
stimulus_condition = ['go', 'nogo']
response_condition = ['correct', 'incorrect']
mind_condition = ['ontask', 'offtask']
conditions_of_interest = ['go/correct/ontask', 'go/correct/offtask']
offtask_metrics = ['mean', 'median', 'quartiles', 'tertiles', 'highlow']

Packages loaded


In [2]:
import os
import mne
from joblib import Parallel, delayed
import re
import pickle
import pandas as pd
import numpy as np


class PSDProcessor:
    def __init__(self, root, tasks, metrics, data="eeg", distance=5, n_jobs=4):
        self.root = root
        self.derivatives_folder = os.path.join(root, "derivatives_nico")
        self.tasks = tasks
        self.metrics = metrics
        self.data = data
        self.distance = distance
        self.n_jobs = n_jobs

    def classify_all_metrics(self, subject_epochs):
        """
        Classify epochs with multiple metrics.
        """
        classified_epochs_dict = {}
        for metric in self.metrics:
            classified_epochs_dict[metric] = classify_onoff_epochs(subject_epochs.copy(), split=metric)
        return classified_epochs_dict

    def process_subject_for_metrics(self, subject):
        """
        Process and classify epochs for a single subject.
        """
        epochs_tasks = []
        for task in self.tasks:
            try:
                epochs, events = read_epochs(self.derivatives_folder, subject, task, self.data, desc="autoPreproc")
                epochs_tasks.append(epochs.copy())
            except Exception as e:
                print(f"Skipping {subject} {task}: {e}")

        if not epochs_tasks:
            print(f"No data for subject {subject}")
            return None

        try:
            epochs_concat = mne.concatenate_epochs(epochs_tasks)
            filtered_epochs = filter_epochs_by_distance_to_probe(epochs_concat, self.distance)
        except Exception as e:
            print(f"Failed concatenating or filtering epochs for subject {subject}: {e}")
            return None
        
        classified_epochs_dict = self.classify_all_metrics(filtered_epochs)

        return classified_epochs_dict
    
    def generate_save_epochs_psd_allmetrics(self, subject, method = 'multitaper', tmin=None, tmax=None,):
        """
        Generate and save PSDs for different metrics.
        """
        
        try:
            classified_epochs_dict = self.process_subject_for_metrics(subject)
            for metric, epochs_classified in classified_epochs_dict.items():
                epoch_psds = epochs_classified.compute_psd(method = method, tmin=tmin, tmax=tmax)

                save_psd_epochs(epoch_psds, self.derivatives_folder, subject, self.data, desc=metric)
                print(f"PSDs saved for subject {subject} using metric {metric}")
        except Exception as e:
            print(f"Failed processing subject {subject}: {e}")
            
    def process_epochs_psd_subjects_parallel(self, subjects,  method = 'multitaper', fmin = 0.5, fmax = 40, tmin=None, tmax=None):
        """
        Parallelized processing for multiple subjects.
        """
        
        Parallel(n_jobs=self.n_jobs)(
            delayed(self.generate_save_epochs_psd_allmetrics)(subject,method = method, tmin=tmin, tmax=tmax) for subject in subjects
        )


    # def compute_aggregated_psds(self, subjects, conditions_of_interest, metrics = ['highlow'],roi = None, aggregate='probe'):
    #     """
    #     Compute PSD data aggregated by condition or probe for LMM analysis.

    #     Parameters:
    #     - participant_epochs: dict, epochs data for each participant
    #     - subjects: list, participants to process
    #     - conditions_of_interest: list, conditions (e.g., ['on-task', 'off-task'])
    #     - roi: list, ROI channels
    #     - freq_bins: list of tuples, each defining a frequency range (start, end) in Hz
    #     - aggregate: str, either 'condition' or 'probe' for different aggregation levels

    #     Returns:
    #     - pd.DataFrame with aggregated PSD data for LMM analysis
    #     """
    #     data_list = []
        
    #     for metric in metrics:
    #         if aggregate == 'condition':
    #             for subject in subjects:
    #                 for condition in conditions_of_interest:
    #                     epochs = read_psd_epochs(self.derivatives_folder, subject, self.data, desc=metric)

    #                     if epochs is None:
    #                         continue  # Skip if no epochs for this condition

    #                     psds, freqs = self.compute_psds(epochs)

    #                     if aggregate == 'condition':
    #                         psds_avg = psds.mean(axis=0)  # Average across epochs
    #                         picks = mne.pick_channels(epochs.info['ch_names'], roi)

    #                         for fmin, fmax in freq_bins:
    #                             freq_mask = (freqs >= fmin) & (freqs <= fmax)
    #                             bin_data = psds_avg[picks][:, freq_mask].mean(axis=1).mean()

    #                             data_list.append({
    #                                 'participant': subject,
    #                                 'condition': condition,
    #                                 'freq_bin': (fmin, fmax),
    #                                 'mean_power': bin_data,
    #                             })

    #                 elif aggregate == 'probe':
    #                     for epoch_idx, epoch_psd in enumerate(psds):
    #                         picks = mne.pick_channels(epochs.info['ch_names'], roi)

    #                         for fmin, fmax in freq_bins:
    #                             freq_mask = (freqs >= fmin) & (freqs <= fmax)
    #                             bin_data = epoch_psd[picks][:, freq_mask].mean(axis=1).mean()

    #                             data_list.append({
    #                                 'participant': subject,
    #                                 'probe': f'epoch{epoch_idx}',
    #                                 'condition': condition,
    #                                 'freq_bin': (fmin, fmax),
    #                                 'mean_power': bin_data,
    #                             })

    #             return pd.DataFrame(data_list)
            
    def aggregate_full_psds_per_condition(
        self, subjects, conditions_of_interest, filename='all_psd_data_conditions.csv'
    ):
        """
        Load previously saved PSD epochs and output a DataFrame with:
        participant, metric, epoch (probe), condition, channel, frequency, power.

        This version loops over conditions, using MNE’s indexing (epochs[condition]),
        ensuring we only get epochs that match a particular condition.

        Parameters:
        - subjects: list of subject IDs
        - conditions_of_interest: list of conditions (strings) to include
        - metrics: list of metrics used to label the PSD data
        - filename: output filename for the CSV

        Returns:
        - A pandas DataFrame containing all PSD data in a long format.
        """

        data_records = []

        for metric in self.metrics:
            for subject in subjects:
                # Load the PSD epochs
                epochs = self.read_psd_epochs_wrapper(subject, metric)
                if epochs is None:
                    continue

                freqs = epochs.freqs
                ch_names = epochs.ch_names

                # Iterate over each condition in conditions_of_interest
                for condition in conditions_of_interest:
                    try:
                        condition_epochs = epochs[condition]
                    except KeyError:
                        # If condition is not present in these epochs, skip
                        continue

                    psd_data = condition_epochs.get_data()  # (n_epochs_cond, n_channels, n_freqs)
                    n_epochs_cond, n_channels, n_freqs = psd_data.shape

                    # We want to keep track of the epoch indices relative to this condition subset.
                    # The indexing with epochs[condition] returns a subset of epochs.
                    # To keep track of the original indices if needed, you can use
                    # condition_epochs.selection. For now, we’ll just use a running index.
                    for ep_idx in range(n_epochs_cond):
                        epoch_psd = psd_data[ep_idx, :, :]  # (n_channels, n_freqs)
                        for ch_idx, ch_name in enumerate(ch_names):
                            for f_idx, f_val in enumerate(freqs):
                                power_val = epoch_psd[ch_idx, f_idx]
                                data_records.append({
                                    'participant': subject,
                                    'metric': metric,
                                    'probe': ep_idx,
                                    'condition': condition,
                                    'channel': ch_name,
                                    'frequency': f_val,
                                    'power': power_val
                                })

        df = pd.DataFrame(data_records)
        out_file = os.path.join(self.derivatives_folder, filename)
        df.to_csv(out_file, index=False)
        return df

    def read_psd_epochs_wrapper(self, subject, metric):
        """
        Wrapper to call your read_psd_epochs function.
        Adjust this as needed based on your actual function signature.
        """
        try:
            # Assuming read_psd_epochs signature: read_psd_epochs(folder, subject, data, desc)
            epochs = read_psd_epochs(self.derivatives_folder, subject, self.data, desc=metric)
            return epochs
        except Exception as e:
            print(f"Could not read PSD epochs for {subject} with metric {metric}: {e}")
            return None

: 

In [ ]:
psd_processor = PSDProcessor(root, tasks, offtask_metrics, n_jobs= -1)

print("Processing subjects and generating PSDs...")
psd_processor.generate_save_epochs_psd_allmetrics('04', tmin=0, tmax=1)
# psd_processor.process_epochs_psd_subjects_parallel(subject, tmin=0, tmax=1)

# # conditions_of_interest = ['go/correct/ontask', 'go/correct/offtask']
# # posterior_roi = ['C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']
# # freq_bins = [(8, 12), (13, 30)]

# # psd_processor.compute_and_save_grand_averages(subjects, conditions_of_interest, posterior_roi, freq_bins)

# # print("Processing completed.")
# # /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-05/eeg/sub-05__task-Sart1_desc-autoPreproc_eeg.fif 

Processing subjects and generating PSDs...
Reading /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart1_desc-autoPreproc_eeg.fif ...
    Found the data of interest:
        t =    -300.00 ...    1200.00 ms
        0 CTF compensation matrices available
Not setting metadata
349 matching events found
No baseline correction applied
0 projection items activated
Loaded epochs from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart1_desc-autoPreproc_eeg.fif
Loaded events from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart1_desc-autoPreproc_events.tsv
Reading /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart2_desc-autoPreproc_eeg.fif ...
    Found the data of interest:
        t =    -300.00 ...    1200.00 ms
        0 CTF compensation matrices available


/Users/nicolas.bruno/depressed_mindwandering/Spectral analysis/../utils/bids_compliance.py:227: RuntimeWarning: This filename (/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart1_desc-autoPreproc_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(bids_path, preload=True)
/Users/nicolas.bruno/depressed_mindwandering/Spectral analysis/../utils/bids_compliance.py:227: RuntimeWarning: This filename (/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart2_desc-autoPreproc_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(bids_path, preload=True)


Not setting metadata
417 matching events found
No baseline correction applied
0 projection items activated
Loaded epochs from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart2_desc-autoPreproc_eeg.fif
Loaded events from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart2_desc-autoPreproc_events.tsv
Reading /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart3_desc-autoPreproc_eeg.fif ...
    Found the data of interest:
        t =    -300.00 ...    1200.00 ms
        0 CTF compensation matrices available
Not setting metadata
422 matching events found
No baseline correction applied
0 projection items activated
Loaded epochs from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart3_desc-autoPreproc_eeg.fif
Loaded events from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart3_desc-autoPreproc_events.tsv
Reading /Volumes/ce

/Users/nicolas.bruno/depressed_mindwandering/Spectral analysis/../utils/bids_compliance.py:227: RuntimeWarning: This filename (/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart3_desc-autoPreproc_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(bids_path, preload=True)
/Users/nicolas.bruno/depressed_mindwandering/Spectral analysis/../utils/bids_compliance.py:227: RuntimeWarning: This filename (/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart4_desc-autoPreproc_eeg.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(bids_path, preload=True)


Not setting metadata
442 matching events found
No baseline correction applied
0 projection items activated
Loaded epochs from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart4_desc-autoPreproc_eeg.fif
Loaded events from: /Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-04/eeg/sub-04__task-Sart4_desc-autoPreproc_events.tsv
Not setting metadata
1630 matching events found
Applying baseline correction (mode: mean)


/var/folders/12/vc8ry6qs78n5ks6ypt4cpx580000gq/T/ipykernel_8694/4122868307.py:46: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  epochs_concat = mne.concatenate_epochs(epochs_tasks)


Effective window size : 1.001 (s)
PSDs saved for subject 04 using metric mean
Effective window size : 1.001 (s)
PSDs saved for subject 04 using metric median
Effective window size : 1.001 (s)
PSDs saved for subject 04 using metric quartiles
Effective window size : 1.001 (s)
PSDs saved for subject 04 using metric tertiles
Effective window size : 1.001 (s)
PSDs saved for subject 04 using metric highlow


In [ ]:
psd_processor = PSDProcessor(root, tasks, offtask_metrics, n_jobs= 5)
psd_processor.aggregate_full_psds_per_condition(subjects, conditions_of_interest)


Could not read PSD epochs for 07 with metric mean: [Errno 2] No such file or directory: '/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-07/eeg/sub-07_psds_desc-mean.pkl'
Could not read PSD epochs for 11 with metric mean: [Errno 2] No such file or directory: '/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-11/eeg/sub-11_psds_desc-mean.pkl'
Could not read PSD epochs for 16 with metric mean: [Errno 2] No such file or directory: '/Volumes/cenir/analyse/meeg/CYBERSART/derivatives_nico/sub-16/eeg/sub-16_psds_desc-mean.pkl'
